# Russian River Step 3 -- 2D mesh

Form the 2D mesh, elevate via a DEM, condition.

In [ ]:
%load_ext autoreload
%autoreload 2
#%matplotlib tk
%matplotlib ipympl

In [ ]:
# setting up logging first or else it gets preempted by another package
import watershed_workflow.io
watershed_workflow.io.setupLogging(1)

In [ ]:
import os,sys
import logging
import numpy as np
from matplotlib import pyplot as plt
import pickle
import shapely
import pandas as pd
import geopandas as gpd
pd.options.display.max_columns = None

import watershed_workflow 
import watershed_workflow.utils
import watershed_workflow.sources
import watershed_workflow.mesh
import watershed_workflow.sources.standard_names as names

# set the default figure size for notebooks
plt.rcParams["figure.figsize"] = (8, 6)

## Input: Parameters and other source data

In [ ]:
# Force Watershed Workflow to pull data from this directory rather than a shared data directory.
# This picks up the Coweeta-specific datasets set up here to avoid large file downloads for 
# demonstration purposes.
#
def splitPathFull(path):
    """
    Splits an absolute path into a list of components such that
    os.path.join(*splitPathFull(path)) == path
    """
    parts = []
    while True:
        head, tail = os.path.split(path)
        if head == path:  # root on Unix or drive letter with backslash on Windows (e.g., C:\)
            parts.insert(0, head)
            break
        elif tail == path:  # just a single file or directory
            parts.insert(0, tail)
            break
        else:
            parts.insert(0, tail)
            path = head
    return parts

cwd = splitPathFull(os.getcwd())
assert cwd[-1] == 'workflow'
cwd = cwd[:-1]

# Note, this directory is where downloaded data will be put as well
data_dir = os.path.join(*(cwd + ['input_data',]))
def toInput(filename):
    return os.path.join(data_dir, filename)

output_dir = os.path.join(*(cwd + ['output_data',]))
output_filenames = dict()
def fromOutput(filename):
    return os.path.join(output_dir, filename)    

def toOutput(role, filename):
    output_filenames[role] = filename
    return fromOutput(filename)

# check output and input dirs exist
if not os.path.isdir(data_dir):
    os.makedirs(data_dir, exist_ok=True)
if not os.path.isdir(output_dir):
    os.makedirs(output_dir, exist_ok=True)
       

In [ ]:
# Set the data directory to the local space to get the locally downloaded files
# REMOVE THIS CELL for general use outside fo Coweeta
watershed_workflow.utils.setDataDirectory(data_dir)


In [ ]:
## Parameters cell -- this provides all parameters that can be changed via pipelining to generate a new watershed. 
name = 'RussianRiver'
hucs = ['18010110'] # a list of HUCs to run

# -- mesh triangle refinement control
refine_d0 = 200
refine_d1 = 600

refine_L0 = 125
refine_L1 = 300

refine_A0 = refine_L0**2 / 2
refine_A1 = refine_L1**2 / 2

# smooth angles
min_angle = 20

# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs = watershed_workflow.crs.default_crs

## Reload data

In [ ]:
with open(fromOutput('02_watersheds.pickle'), 'rb') as fid:
    watersheds = pickle.load(fid)

reaches = gpd.read_parquet(fromOutput('02_rivers.parquet'))
rivers = watershed_workflow.hydro.createRivers(reaches, method='native')


In [ ]:
fig, ax = plt.subplots()
reaches[names.BANKFULL_WIDTH].hist(bins=20)
plt.show()
print(reaches[names.BANKFULL_WIDTH].min())
print(reaches[names.BANKFULL_WIDTH].max())
print(any(reaches[names.BANKFULL_WIDTH].isna()))

reaches[reaches[names.BANKFULL_WIDTH] == 0.0]

## Generate the mesh


In [ ]:
def riverWidth(reach):
    width = reach[names.BANKFULL_WIDTH]
    if width == 0.0:
        if len(reach.children) == 1:
            return riverWidth(reach.children[0])
        elif reach.parent is not None:
            return riverWidth(reach.parent)
        elif len(reach.children) > 1:
            return max(riverWidth(child) for child in reach.children)
        else:
            raise RuntimeError(f'Cannot get width for reach {reach['ID']}')
    elif width < 5:
        return 5
    else:
        return width



            
m2, areas, dists = watershed_workflow.tessalateRiverAligned(watersheds, rivers, river_width = riverWidth,
                                             refine_min_angle = min_angle, refine_distance = [refine_d0, refine_A0, refine_d1, refine_A1],
                                             diagnostics=True, debug=False, triangulate=True)

In [ ]:
print([riverWidth(child) for child in rivers[0].children])

In [ ]:
# Add basic labeled sets, including an outlet
outlet_edge = watershed_workflow.mesh.Edge(rivers[0]['elems'][-1][0],rivers[0]['elems'][-1][-1])
watersheds.df[watershed_workflow.sources.standard_names.OUTLET] = [shapely.geometry.Point([m2.edge_centroids[outlet_edge]]),]

# add a single region for the lone outlet edge
watershed_workflow.mesh.addOutletRegion(m2, outlet_edge, 'outlet')

# add regions for each polygon
watershed_workflow.mesh.addWatershedAndOutletRegions(m2, watersheds, 500)

# add regions for each stream order
watershed_workflow.mesh.addStreamOrderRegions(m2, rivers)

In [ ]:
print('2D labeled sets')
print('---------------')
for ls in m2.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : {len(ls.ent_ids)} : "{ls.name}"')

In [ ]:
# save to file -- we will use this for testing pitfilling algorithms
with open(toOutput('m2_no_elevation', '03a_m2_no_elevation.pickle'), 'wb') as fid:
    pickle.dump(m2, fid)

In [ ]:
# save this final version of rivers and watersheds to disk
with open(toOutput('watershed_polys2', '03a_watersheds.pickle'), 'wb') as fid:
    pickle.dump(watersheds, fid)

river_df = gpd.GeoDataFrame(pd.concat([r.to_dataframe() for r in rivers]), crs=crs)
river_df.to_parquet(toOutput('rivers2', '03a_rivers.parquet'))

In [ ]:
# save output filenames
with open(toOutput('03a_output_filenames', '03a_output_filenames.txt'), 'wb') as fid:
    pickle.dump(output_filenames, fid)